<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/02%20bigquery/02_ENARES_2024_STAGE2_load_crs04_sav_to_bigquery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 02_ENARES_2024_STAGE2_load_crs04_sav_to_bigquery.ipynb
# Stage 2 - Load CRS04 .sav files to BigQuery raw tables
# ============================================================

!pip install -q google-cloud-bigquery pandas pyreadstat pandas-gbq pyarrow

from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
import pandas as pd
import os
import pyreadstat

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = input("Enter your Google Cloud PROJECT_ID: ").strip()
LOCATION = "US"

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
SAV_DIR = f"{ROOT_DRIVE}/01BasesDatosPrimarias"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"

os.makedirs(LOG_DIR, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

In [ ]:
# ============================================================
# 1. Verify raw dataset exists
# ============================================================

raw_dataset_id = f"{PROJECT_ID}.enares2024_crs04_raw"

try:
    raw_dataset = client.get_dataset(raw_dataset_id)
    print(f"Raw dataset exists: {raw_dataset_id}")
except Exception as e:
    raise RuntimeError(
        "Raw dataset does not exist. Run Notebook 1 first."
    ) from e

In [ ]:
# ============================================================
# 2. Map CRS04 .sav files to BigQuery raw tables
# ============================================================

table_mapping = pd.DataFrame([
    {
        "module": "CRS04",
        "chapter": "CAP100",
        "source_file": "19_CRS04_CAP100.sav",
        "target_table": "raw_crs04_cap100"
    },
    {
        "module": "CRS04",
        "chapter": "CAP200",
        "source_file": "20_CRS04_CAP200.sav",
        "target_table": "raw_crs04_cap200"
    },
    {
        "module": "CRS04",
        "chapter": "CAP248",
        "source_file": "21_CRS04_CAP248.sav",
        "target_table": "raw_crs04_cap248"
    },
    {
        "module": "CRS04",
        "chapter": "CAP300",
        "source_file": "22_CRS04_CAP300.sav",
        "target_table": "raw_crs04_cap300"
    },
])

table_mapping["source_path"] = table_mapping["source_file"].apply(
    lambda filename: f"{SAV_DIR}/{filename}"
)

table_mapping["raw_dataset"] = "enares2024_crs04_raw"

mapping_output = f"{LOG_DIR}/ENARES_2024_STAGE2_crs04_bigquery_table_mapping.csv"
table_mapping.to_csv(mapping_output, index=False)

print(f"Mapping saved to: {mapping_output}")
display(table_mapping)

In [ ]:

# ============================================================
# 3. Verify source files exist before loading
# If files are inside subfolders, auto-detect their real paths
# ============================================================

from pathlib import Path
import os

base_dir = Path(SAV_DIR)

if not base_dir.exists():
    raise FileNotFoundError(f"SAV_DIR does not exist: {SAV_DIR}")

def find_sav_file(source_file):
    direct_path = base_dir / source_file

    if direct_path.exists():
        return str(direct_path)

    matches = list(base_dir.rglob(source_file))

    if len(matches) == 1:
        return str(matches[0])

    if len(matches) == 0:
        return None

    raise ValueError(
        f"More than one file named {source_file} found under {SAV_DIR}: {matches}"
    )

table_mapping["source_path"] = table_mapping["source_file"].apply(find_sav_file)

table_mapping["file_exists"] = table_mapping["source_path"].apply(
    lambda path: path is not None and os.path.exists(path)
)

table_mapping["file_size_bytes"] = table_mapping["source_path"].apply(
    lambda path: os.path.getsize(path) if path is not None and os.path.exists(path) else None
)

source_check_output = f"{LOG_DIR}/ENARES_2024_STAGE2_source_file_check.csv"
table_mapping.to_csv(source_check_output, index=False)

display(table_mapping)

if not table_mapping["file_exists"].all():
    raise FileNotFoundError(
        "At least one CRS04 .sav file is missing. Check file names or subfolder location."
    )

print("All four CRS04 .sav files exist.")
print(f"Source check saved to: {source_check_output}")

In [ ]:
# ============================================================
# 4. Load .sav files to BigQuery raw tables
# IMPORTANT: apply_value_formats=False preserves original SPSS codes.
# Do not recode, filter, merge, or create analytical variables here.
# ============================================================

raw_inventory = []
schema_records = []

for _, row in table_mapping.iterrows():
    source_path = row["source_path"]
    target_table = row["target_table"]
    full_table_id = f"{PROJECT_ID}.enares2024_crs04_raw.{target_table}"

    print(f"\nLoading: {row['source_file']} -> {full_table_id}")

    df, meta = pyreadstat.read_sav(
        source_path,
        apply_value_formats=False
    )

    sav_columns = list(df.columns)

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        autodetect=True
    )

    load_job = client.load_table_from_dataframe(
        df,
        full_table_id,
        job_config=job_config
    )

    load_job.result()

    bq_table = client.get_table(full_table_id)
    bq_columns = [field.name for field in bq_table.schema]

    raw_inventory.append({
        "source_file": row["source_file"],
        "chapter": row["chapter"],
        "target_table": target_table,
        "full_table_id": full_table_id,
        "sav_rows": len(df),
        "sav_columns": len(sav_columns),
        "bq_rows": bq_table.num_rows,
        "bq_columns": len(bq_columns),
        "loaded_at_utc": datetime.now(timezone.utc).isoformat()
    })

    schema_records.append({
        "source_file": row["source_file"],
        "chapter": row["chapter"],
        "target_table": target_table,
        "sav_column_count": len(sav_columns),
        "bq_column_count": len(bq_columns),
        "column_count_match": len(sav_columns) == len(bq_columns),
        "missing_columns_from_bigquery": ", ".join(sorted(set(sav_columns) - set(bq_columns))),
        "extra_columns_in_bigquery": ", ".join(sorted(set(bq_columns) - set(sav_columns))),
        "checked_at_utc": datetime.now(timezone.utc).isoformat()
    })

raw_inventory = pd.DataFrame(raw_inventory)
schema_validation = pd.DataFrame(schema_records)

raw_inventory_output = f"{LOG_DIR}/ENARES_2024_STAGE2_raw_table_inventory.csv"
schema_output = f"{LOG_DIR}/ENARES_2024_STAGE2_schema_validation.csv"

raw_inventory.to_csv(raw_inventory_output, index=False)
schema_validation.to_csv(schema_output, index=False)

print(f"\nRaw inventory saved to: {raw_inventory_output}")
print(f"Schema validation saved to: {schema_output}")

display(raw_inventory)
display(schema_validation)

In [ ]:
# ============================================================
# 5. Validate row counts: .sav versus BigQuery
# ============================================================

rowcount_validation = raw_inventory.copy()

rowcount_validation["rowcount_match"] = (
    rowcount_validation["sav_rows"] == rowcount_validation["bq_rows"]
)

rowcount_output = f"{LOG_DIR}/ENARES_2024_STAGE2_rowcount_validation.csv"
rowcount_validation.to_csv(rowcount_output, index=False)

display(rowcount_validation)

if not rowcount_validation["rowcount_match"].all():
    raise ValueError(
        "Rowcount mismatch detected. Do not close Stage 2 until this is fixed."
    )

print(f"Rowcount validation saved to: {rowcount_output}")
print("All row counts match.")

In [ ]:
# ============================================================
# 6. Validate schema results
# ============================================================

display(schema_validation)

if not schema_validation["column_count_match"].all():
    raise ValueError(
        "Schema mismatch detected. Do not close Stage 2 until this is reviewed."
    )

print("All column counts match.")

In [ ]:
# ============================================================
# 7. BigQuery raw table existence check
# ============================================================

raw_tables_sql = f"""
SELECT table_id AS table_name, row_count
FROM `{PROJECT_ID}.enares2024_crs04_raw.__TABLES__`
WHERE table_id IN (
  'raw_crs04_cap100',
  'raw_crs04_cap200',
  'raw_crs04_cap248',
  'raw_crs04_cap300'
)
ORDER BY table_id
"""

raw_tables_check = client.query(raw_tables_sql).result().to_dataframe()

raw_tables_check_output = f"{LOG_DIR}/ENARES_2024_STAGE2_bigquery_raw_tables_check.csv"
raw_tables_check.to_csv(raw_tables_check_output, index=False)

display(raw_tables_check)

print(f"BigQuery raw table check saved to: {raw_tables_check_output}")

In [ ]:
# ============================================================
# 8. Final acceptance check for Notebook 2
# ============================================================

expected_outputs = [
    "ENARES_2024_STAGE2_crs04_bigquery_table_mapping.csv",
    "ENARES_2024_STAGE2_source_file_check.csv",
    "ENARES_2024_STAGE2_raw_table_inventory.csv",
    "ENARES_2024_STAGE2_rowcount_validation.csv",
    "ENARES_2024_STAGE2_schema_validation.csv",
    "ENARES_2024_STAGE2_bigquery_raw_tables_check.csv",
]

for filename in expected_outputs:
    path = f"{LOG_DIR}/{filename}"
    print(filename, "OK" if os.path.exists(path) else "MISSING")

print("\nNotebook 2 completed successfully.")
print("Remember: Stage 2 loaded raw tables only. No merge, no recoding, no indicators.")